# LangChain 基本範例

LangChain 可以把 Prompt、LLM 與輸出處理器組合成可重複使用的 Chain。

這份 notebook 將介紹：

1. 直接呼叫 LLM。
2. 使用 `PromptTemplate` 建立提示詞。
3. 使用 `StrOutputParser` 取得字串結果。
4. 使用 LCEL 的 `|` 組成 Chain。
5. 把兩條 Chain 組成簡單的多步驟流程。

## 1. Import libraries

In [ ]:
import os

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

## 2. 建立 Chat Model

請先在系統環境變數中設定 `OPENAI_API_KEY`。也可以用 `OPENAI_MODEL` 更換模型；若未設定，預設使用 `gpt-4o-mini`。

In [ ]:
llm_options = {
    "api_key": os.environ["OPENAI_API_KEY"],
    "model": os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    "temperature": 0.0,
}

if os.getenv("OPENAI_BASE_URL"):
    llm_options["base_url"] = os.environ["OPENAI_BASE_URL"]

llm = ChatOpenAI(**llm_options)

## 3. 直接呼叫 LLM

`invoke()` 是最基本的執行方法。Chat Model 回傳的是 `AIMessage`，回答文字位於 `.content`。

In [ ]:
response = llm.invoke("請用一句話說明什麼是 LangChain。")

print(type(response))
print(response.content)

## 4. 建立 PromptTemplate

Prompt Template 將固定指示與變動資料分開。`{topic}` 與 `{audience}` 是執行時才填入的變數。

In [ ]:
prompt = PromptTemplate.from_template(
    "你是一位耐心的老師。請用三個條列重點，向{audience}解釋{topic}。"
)

In [ ]:
formatted_prompt = prompt.invoke(
    {"topic": "大型語言模型", "audience": "初學者"}
)

print(formatted_prompt.text)

## 5. 建立 Output Parser

LLM 原本回傳 `AIMessage`。`StrOutputParser` 會取出其中的文字，讓 Chain 最後直接得到 Python 字串。

In [ ]:
parser = StrOutputParser()

## 6. 不使用 LCEL 的寫法

以下程式明確呈現資料依序經過 Prompt、LLM、Parser。

In [ ]:
prompt_value = prompt.invoke(
    {"topic": "LangChain", "audience": "Python 初學者"}
)
ai_message = llm.invoke(prompt_value)
answer = parser.invoke(ai_message)

print(answer)

## 7. 使用 LCEL 組成 Chain

LCEL（LangChain Expression Language）使用 `|` 表示資料流向：

```text
輸入字典 → PromptTemplate → ChatOpenAI → StrOutputParser → 字串
```

In [ ]:
chain = prompt | llm | parser

In [ ]:
result = chain.invoke(
    {"topic": "LangChain", "audience": "Python 初學者"}
)

print(result)
print(type(result))

`|` 並不是單純的字串串接。Prompt、LLM 與 Parser 都實作了 LangChain 的 Runnable 介面，因此可以組合並統一使用 `invoke()`、`batch()` 和 `stream()`。

## 8. Batch 與 Stream

同一條 Chain 可以一次處理多個輸入，也可以逐段輸出結果。

In [ ]:
batch_results = chain.batch(
    [
        {"topic": "API", "audience": "初學者"},
        {"topic": "Prompt", "audience": "初學者"},
    ]
)

for item in batch_results:
    print(item)
    print("---")

In [ ]:
for chunk in chain.stream(
    {"topic": "LCEL", "audience": "初學者"}
):
    print(chunk, end="", flush=True)

## 9. 簡單的多步驟 Chain

接下來把第一條 Chain 產生的教學內容交給第二個 Prompt，請 LLM 根據內容產生測驗題：

```text
主題 → 產生教學內容 → 產生測驗題 → 字串結果
```

In [ ]:
lesson_prompt = PromptTemplate.from_template(
    "請用簡單的方式介紹 {topic}，內容限制在 150 字以內。"
)
lesson_chain = lesson_prompt | llm | parser

quiz_prompt = PromptTemplate.from_template(
    "根據以下教學內容，產生一題四選一測驗，並在最後提供答案：\n\n{lesson}"
)

multi_step_chain = (
    {"lesson": lesson_chain}
    | quiz_prompt
    | llm
    | parser
)

In [ ]:
quiz = multi_step_chain.invoke({"topic": "LangChain"})
print(quiz)

## 10. 查看 Chain 結構

In [ ]:
multi_step_chain.get_graph().print_ascii()

## 重點整理

| 元件 | 用途 |
|---|---|
| `PromptTemplate` | 將輸入資料填入提示詞 |
| `ChatOpenAI` | 將提示詞送給語言模型 |
| `StrOutputParser` | 將 `AIMessage` 轉成字串 |
| `|` | 使用 LCEL 串接 Runnable |
| `invoke()` | 執行一次 |
| `batch()` | 一次處理多筆輸入 |
| `stream()` | 逐段取得輸出 |

最基本的 LangChain 寫法就是：

```python
chain = prompt | llm | parser
result = chain.invoke(input_data)
```